In [85]:
# =============================================================================
# 02_daigt_data_engineering.py
# DAIGT Dataset — Load, Validate, Clean, Merge with HC3
# Detecting AI-Generated Text | MSc Data Science | Group Project
# =============================================================================
# Run order:  01_hc3_data_engineering.py FIRST, then this file
# Requires:   data/hc3_clean.csv  (output of file 01)
# Output:     data/daigt_clean.csv         — cleaned DAIGT only
#             data/merged.csv              — HC3 + DAIGT combined
#             data/train.csv               — 70% stratified split
#             data/val.csv                 — 15% stratified split
#             data/test.csv                — 15% stratified split (never touch until final eval)
# =============================================================================

import os
import re
import string
import pandas as pd
from sklearn.model_selection import train_test_split
 
# langdetect for filtering non-English rows — same as file 01
from langdetect import detect, LangDetectException
 
# ── Reproducibility ───────────────────────────────────────────────────────────
# Must match the seed used in file 01 so every team member gets identical splits.
RANDOM_SEED = 42
 
# ── Label schema ─────────────────────────────────────────────────────────────
# Identical to file 01 — integer labels throughout.
LABEL_HUMAN = 0
LABEL_AI    = 1
 
# ── Dataset identifier ────────────────────────────────────────────────────────
DATASET_NAME = "DAIGT"
 
# ── Minimum text length ───────────────────────────────────────────────────────
# Same threshold as file 01 for consistency across both datasets.
MIN_WORD_COUNT = 10
 
# ── Class balance threshold ───────────────────────────────────────────────────
# If the minority class drops below this fraction of the total merged dataset,
# we apply random undersampling of the majority class to restore balance.
# 0.35 means we act if the minority class is less than 35% of the data.
MIN_CLASS_RATIO = 0.35
 
# ── Train / Val / Test split ratios ───────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15   # must equal 1 - TRAIN_RATIO - VAL_RATIO
 
# ── File paths ────────────────────────────────────────────────────────────────
DATA_DIR        = "/Users/yashaswini11/Desktop/Team_project/project"
HC3_CLEAN_FILE  = os.path.join(DATA_DIR, "hc3_clean.csv")       # input  (from file 01)
DAIGT_RAW_FILE  = os.path.join(DATA_DIR, "train_v4_drcat_01.csv")  # input  (Kaggle download)
DAIGT_OUT_FILE  = os.path.join(DATA_DIR, "daigt_clean.csv")     # output (DAIGT only)
MERGED_FILE     = os.path.join(DATA_DIR, "merged.csv")           # output (HC3 + DAIGT)
TRAIN_FILE      = os.path.join(DATA_DIR, "train.csv")            # output
VAL_FILE        = os.path.join(DATA_DIR, "val.csv")              # output
TEST_FILE       = os.path.join(DATA_DIR, "test.csv")             # output
 
os.makedirs(DATA_DIR, exist_ok=True)
 
# ── DAIGT column names ────────────────────────────────────────────────────────
# These are the exact column names in the DAIGT V4 CSV from Kaggle.
DAIGT_TEXT_COL  = "text"
DAIGT_LABEL_COL = "label"   # 0 = human, 1 = AI — matches our schema directly


In [86]:
print(DAIGT_TEXT_COL)
print(DAIGT_LABEL_COL)

text
label


In [87]:
 
# =============================================================================
# SECTION 1 — Load HC3 Clean (output of file 01)
# =============================================================================
# We load the already-cleaned HC3 data produced by file 01.
# This file is our baseline — DAIGT must be cleaned to the same standard
# before the two datasets can be merged.
# =============================================================================
 
print("=" * 60)
print("STEP 1: Loading cleaned HC3 data")
print("=" * 60)
 
# Guard: fail immediately with a clear message if file 01 has not been run yet.
if not os.path.exists(HC3_CLEAN_FILE):
    raise FileNotFoundError(
        f"\n[ERROR] {HC3_CLEAN_FILE} not found.\n"
        "Please run 01_hc3_data_engineering.py first before running this file."
    )
 
hc3_df = pd.read_csv(HC3_CLEAN_FILE, encoding='utf-8-sig')
 
print(f"  HC3 rows loaded       : {len(hc3_df):,}")
print(f"  HC3 columns           : {hc3_df.columns.tolist()}")
print(f"\n  HC3 label distribution:")
hc3_counts = hc3_df['label'].value_counts().sort_index()
for label, count in hc3_counts.items():
    name = "Human" if label == 0 else "AI"
    print(f"    {label} ({name}) : {count:,}  ({count/len(hc3_df)*100:.1f}%)")


STEP 1: Loading cleaned HC3 data
  HC3 rows loaded       : 78,733
  HC3 columns           : ['text', 'label', 'source', 'dataset', 'word_count']

  HC3 label distribution:
    0 (Human) : 52,525  (66.7%)
    1 (AI) : 26,208  (33.3%)


In [88]:
hc3_df.head()

,text,label,source,dataset,word_count
0,"Basically there are many categories of "" Best ...",0,reddit_eli5,HC3,134
1,"If you 're hearing about it , it 's because it...",0,reddit_eli5,HC3,73
2,"One reason is lots of catagories . However , h...",0,reddit_eli5,HC3,62
3,salt is good for not dying in car crashes and ...,0,reddit_eli5,HC3,41
4,"In Minnesota and North Dakota , they tend to u...",0,reddit_eli5,HC3,116


In [89]:
# =============================================================================
# SECTION 2 — Load Raw DAIGT CSV
# =============================================================================
# DAIGT V4 is downloaded from Kaggle:
#   https://www.kaggle.com/datasets/thedrcat/daigt-v4-train-dataset
#
# Kaggle download command (run in your terminal):
#   kaggle datasets download -d thedrcat/daigt-v4-train-dataset -p data/
#   unzip data/daigt-v4-train-dataset.zip -d data/
#
# The CSV has a flat structure — no nested lists, no explosion needed:
#   text      (str) — the raw text sample
#   label (int) — 0 = human-written, 1 = AI-generated
#
# This is the key structural difference from HC3:
#   HC3   → nested lists, implicit labels, must be flattened  (done in file 01)
#   DAIGT → already flat, explicit label column, just clean and align schema
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 2: Loading raw DAIGT CSV")
print("=" * 60)

DAIGT_RAW_FILE = "/Users/yashaswini11/Desktop/Team_project/project/train_v4_drcat_01.csv"
if not os.path.exists(DAIGT_RAW_FILE):
    raise FileNotFoundError(
        f"\n[ERROR] {DAIGT_RAW_FILE} not found.\n"
        "Download the DAIGT V4 dataset from Kaggle:\n"
        "  https://www.kaggle.com/datasets/thedrcat/daigt-v4-train-dataset\n"
        f"Place the CSV at: {DAIGT_RAW_FILE}"
    )
 
daigt_df = pd.read_csv('/Users/yashaswini11/Desktop/Team_project/project/train_v4_drcat_01.csv')
 
print(f"  Raw rows loaded  : {len(daigt_df):,}")
print(f"  Columns          : {daigt_df.columns.tolist()}")
print(f"  dtypes:\n{daigt_df.dtypes.to_string()}")
 
# Quick structural peek — confirm the two essential columns exist
assert DAIGT_TEXT_COL in daigt_df.columns, \
    f"Expected column '{DAIGT_TEXT_COL}' not found. Columns: {daigt_df.columns.tolist()}"
assert DAIGT_LABEL_COL in daigt_df.columns, \
    f"Expected column '{DAIGT_LABEL_COL}' not found. Columns: {daigt_df.columns.tolist()}"
 
print(f"\n  Raw label distribution:")
raw_counts = daigt_df[DAIGT_LABEL_COL].value_counts().sort_index()
for label, count in raw_counts.items():
    name = "Human" if label == 0 else "AI"
    print(f"    {label} ({name}) : {count:,}  ({count/len(daigt_df)*100:.1f}%)")
 
print(f"\n  Sample raw rows:")
print(daigt_df[[DAIGT_TEXT_COL, DAIGT_LABEL_COL]].head(3).to_string())


STEP 2: Loading raw DAIGT CSV
  Raw rows loaded  : 73,573
  Columns          : ['text', 'label', 'prompt_name', 'source', 'RDizzl3_seven', 'model']
  dtypes:
text             object
label             int64
prompt_name      object
source           object
RDizzl3_seven      bool
model            object

  Raw label distribution:
    0 (Human) : 27,370  (37.2%)
    1 (AI) : 46,203  (62.8%)

  Sample raw rows:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [90]:
daigt_df.head()

,text,label,prompt_name,source,RDizzl3_seven,model
0,Phones\n\nModern humans today are always on th...,0,Phones and driving,persuade_corpus,False,human
1,This essay will explain if drivers should or s...,0,Phones and driving,persuade_corpus,False,human
2,Driving while the use of cellular devices\n\nT...,0,Phones and driving,persuade_corpus,False,human
3,Phones & Driving\n\nDrivers should not be able...,0,Phones and driving,persuade_corpus,False,human
4,Cell Phone Operation While Driving\n\nThe abil...,0,Phones and driving,persuade_corpus,False,human


In [91]:

# =============================================================================
# SECTION 3 — Standardise Schema to Match HC3
# =============================================================================
# After file 01, HC3 has these columns:
#   text | label | source | dataset | word_count
#
# daigt_magic_generations.csv has these columns:
#   text | label | prompt_name | source | RDizzl3_seven | model
#
# 'text' and 'label' already match our schema — no renaming needed.
# 'source' already exists and is kept as-is.
# We just need to:
#   1. Add    'dataset' → "DAIGT"  (marks provenance for cross-eval later)
#   2. Drop   'prompt_name', 'RDizzl3_seven', 'model'  (not needed downstream)
#
# Note: 'word_count' is NOT added here — it is computed in Section 8
# after text cleaning, same as HC3. Both datasets have it before the merge.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 3: Standardising schema to match HC3")
print("=" * 60)
 
# Keep only the columns we need — text and label already match, source is preserved
daigt_df = daigt_df[['text', 'label', 'source']].copy()
 
# Add provenance metadata column
daigt_df['dataset'] = DATASET_NAME
 
print(f"  Columns after standardisation : {daigt_df.columns.tolist()}")
print(f"  Schema now matches HC3        : text | label | source | dataset")
 
# Confirm label values are still valid integers 0/1
print(f"  Unique label values           : {sorted(daigt_df['label'].unique().tolist())}")
 
 


STEP 3: Standardising schema to match HC3
  Columns after standardisation : ['text', 'label', 'source', 'dataset']
  Schema now matches HC3        : text | label | source | dataset
  Unique label values           : [0, 1]


In [92]:
daigt_df.head()

,text,label,source,dataset
0,Phones\n\nModern humans today are always on th...,0,persuade_corpus,DAIGT
1,This essay will explain if drivers should or s...,0,persuade_corpus,DAIGT
2,Driving while the use of cellular devices\n\nT...,0,persuade_corpus,DAIGT
3,Phones & Driving\n\nDrivers should not be able...,0,persuade_corpus,DAIGT
4,Cell Phone Operation While Driving\n\nThe abil...,0,persuade_corpus,DAIGT


In [93]:
# =============================================================================
# SECTION 4 — Enforce Correct Data Types - handling NaN values
# =============================================================================
# CSV loading can produce unexpected types — e.g., label read as float64
# if any NaN was present in that column. We enforce types explicitly before
# cleaning to avoid silent errors downstream.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 4: Enforcing data types")
print("=" * 60)
 
# Cast text to string — catches any NaN values that slipped through
daigt_df['text'] = daigt_df['text'].astype(str)

# Cast label to integer — ensures 0/1, not 0.0/1.0
daigt_df['label'] = daigt_df['label'].astype(int)
 
# Cast metadata to string
daigt_df['source']  = daigt_df['source'].astype(str)
daigt_df['dataset'] = daigt_df['dataset'].astype(str)
 
print(f"dtypes after enforcement:\n{daigt_df.dtypes.to_string()}")


STEP 4: Enforcing data types
dtypes after enforcement:
text       object
label       int64
source     object
dataset    object


In [94]:

# =============================================================================
# SECTION 5 — Drop Nulls and Empty Strings
# =============================================================================
# Even though DAIGT is cleaner than HC3 (no nested list explosion),
# null values and empty strings can still exist in the raw CSV.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 5: Dropping nulls and empty strings")
print("=" * 60)
 
before = len(daigt_df)
 
# Drop rows where text or label is NaN
daigt_df = daigt_df.dropna(subset=['text', 'label'])
 
# Drop rows where text is an empty string or only whitespace
daigt_df = daigt_df[daigt_df['text'].str.strip() != '']
 
after = len(daigt_df)
daigt_df = daigt_df.reset_index(drop=True)
 
print(f"  Rows before : {before:,}")
print(f"  Rows after  : {after:,}")
print(f"  Dropped     : {before - after:,}")


STEP 5: Dropping nulls and empty strings
  Rows before : 73,573
  Rows after  : 73,573
  Dropped     : 0


In [95]:

# =============================================================================
# SECTION 6 — Clean Text
# =============================================================================
# We apply the identical clean_text() function used in file 01.
# Keeping the same function ensures both datasets are cleaned to exactly
# the same standard — no subtle differences in whitespace handling or
# URL removal between the two.
#
# Operations:
#   1. Cast to string
#   2. Remove URLs
#   3. Remove HTML tags
#   4. Decode HTML entities
#   5. Collapse whitespace (tabs, newlines → single space)
#   6. Strip leading/trailing whitespace
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 6: Cleaning text")
print("=" * 60)
 
def clean_text(text: str) -> str:
    """
    Clean a single text string for AI detection modelling.
 
    Operations (in order):
        1. Cast to string — handles any NaN or non-string values safely
        2. Remove URLs
        3. Remove HTML tags
        4. Decode common HTML entities
        5. Collapse whitespace (tabs, newlines → single space)
        6. Strip leading/trailing whitespace
 
    Args:
        text: Raw input string (or any type — will be cast to str)
 
    Returns:
        Cleaned string. Empty string if input was null/empty.
    """
    # Cast to string
    text = str(text)
 
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
 
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
 
    # Decode common HTML entities
    text = text.replace('&amp;', '&')
    text = text.replace('&lt;', '<')
    text = text.replace('&gt;', '>')
    text = text.replace('&nbsp;', ' ')
    text = text.replace('&#39;', "'")
    text = text.replace('&quot;', '"')
 
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text)
 
    # Strip edges
    text = text.strip()
 
    return text
 
 
daigt_df['text'] = daigt_df['text'].apply(clean_text)
print(f"  Cleaning applied to {len(daigt_df):,} rows")
 
# Drop any rows that became empty after cleaning (e.g., a row that was only a URL)
before = len(daigt_df)
daigt_df = daigt_df[daigt_df['text'].str.strip() != ''].reset_index(drop=True)
print(f"  Rows that became empty after cleaning and were dropped : {before - len(daigt_df):,}")


STEP 6: Cleaning text
  Cleaning applied to 73,573 rows
  Rows that became empty after cleaning and were dropped : 0


In [96]:
daigt_df.head()

,text,label,source,dataset
0,Phones Modern humans today are always on their...,0,persuade_corpus,DAIGT
1,This essay will explain if drivers should or s...,0,persuade_corpus,DAIGT
2,Driving while the use of cellular devices Toda...,0,persuade_corpus,DAIGT
3,Phones & Driving Drivers should not be able to...,0,persuade_corpus,DAIGT
4,Cell Phone Operation While Driving The ability...,0,persuade_corpus,DAIGT


In [97]:

# =============================================================================
# SECTION 7 — Filter Non-English Text
# =============================================================================
# DAIGT is primarily English but includes some non-English samples depending
# on the version. We apply the same English filter used in file 01.
#
# langdetect.detect() returns a language code: 'en', 'fr', 'de', etc.
# We keep only rows where the detected language is English ('en').
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 7: Filtering non-English text")
print("=" * 60)
 
def is_english(text: str) -> bool:
    """
    Return True if langdetect identifies the text as English.
 
    Args:
        text: Input string to test.
 
    Returns:
        True if English, False for any other language or detection failure.
    """
    try:
        return detect(text) == 'en'
    except LangDetectException:
        # LangDetectException is raised for texts that are too short or
        # contain only symbols/numbers with no detectable language pattern.
        # We conservatively treat these as non-English and drop them.
        return False
 
 
before = len(daigt_df)
english_mask = daigt_df['text'].apply(is_english)
daigt_df = daigt_df[english_mask].reset_index(drop=True)
after = len(daigt_df)
 
print(f"  Rows before language filter : {before:,}")
print(f"  Rows after language filter  : {after:,}")
print(f"  Non-English rows dropped    : {before - after:,}")
 


STEP 7: Filtering non-English text
  Rows before language filter : 73,573
  Rows after language filter  : 73,569
  Non-English rows dropped    : 4


In [98]:

# =============================================================================
# SECTION 8 — Drop Texts Below Minimum Word Count
# =============================================================================
# Same MIN_WORD_COUNT = 10 threshold as file 01.
# We compute word_count here and keep it as a permanent column —
# it becomes a stylometric feature in the feature engineering notebook.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 8: Removing texts shorter than minimum word count")
print("=" * 60)
 
daigt_df['word_count'] = daigt_df['text'].str.split().str.len()
 
before = len(daigt_df)
daigt_df = daigt_df[daigt_df['word_count'] >= MIN_WORD_COUNT].reset_index(drop=True)
after = len(daigt_df)
 
print(f"  Minimum word count threshold : {MIN_WORD_COUNT}")
print(f"  Rows before                  : {before:,}")
print(f"  Rows after                   : {after:,}")
print(f"  Dropped                      : {before - after:,}")
print(f"\n  Word count stats:")
print(daigt_df['word_count'].describe().round(1).to_string())


STEP 8: Removing texts shorter than minimum word count
  Minimum word count threshold : 10
  Rows before                  : 73,569
  Rows after                   : 73,566
  Dropped                      : 3

  Word count stats:
count    73566.0
mean       385.5
std        163.3
min         21.0
25%        276.0
50%        360.0
75%        459.0
max       1656.0


In [99]:
daigt_df.columns.tolist()

['text', 'label', 'source', 'dataset', 'word_count']

In [100]:

# =============================================================================
# SECTION 9 — Remove Duplicate Texts Within DAIGT
# =============================================================================
# We deduplicate DAIGT internally before the cross-dataset deduplication step.
# Doing it in two stages makes it easier to track exactly where duplicates
# are coming from (within-dataset vs cross-dataset).
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 9: Removing duplicate texts within DAIGT")
print("=" * 60)
 
before = len(daigt_df)
daigt_df = daigt_df.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)
after = len(daigt_df)
 
print(f"  Rows before deduplication : {before:,}")
print(f"  Rows after deduplication  : {after:,}")
print(f"  Duplicates removed        : {before - after:,}")


STEP 9: Removing duplicate texts within DAIGT
  Rows before deduplication : 73,566
  Rows after deduplication  : 73,560
  Duplicates removed        : 6


In [101]:
 
# =============================================================================
# SECTION 10 — Schema Assertions for DAIGT
# =============================================================================
# Same assertion pattern as file 01 — automated quality gate before saving.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 10: Running data quality assertions (DAIGT)")
print("=" * 60)
 
# Check 1 — Required columns present
required_cols = {'text', 'label', 'source', 'dataset', 'word_count'}
assert required_cols.issubset(set(daigt_df.columns)), \
    f"Missing columns: {required_cols - set(daigt_df.columns)}"
print("  ✓ All required columns present")
 
# Check 2 — No nulls in critical columns
null_counts = daigt_df[['text', 'label']].isnull().sum()
assert null_counts.sum() == 0, f"Null values found:\n{null_counts}"
print("  ✓ No null values in text or label columns")
 
# Check 3 — Valid labels only
assert daigt_df['label'].isin([0, 1]).all(), \
    "Label column contains values other than 0 and 1"
print("  ✓ All labels are 0 or 1")
 
# Check 4 — Both classes present
# (Updated: train_v4_drcat_01.csv has both human and AI — this is correct)
assert daigt_df['label'].nunique() == 2, \
    "Dataset contains only one class — check raw CSV"
print("  ✓ Both classes (0=Human, 1=AI) present")

# Check 4b — Sanity check on class distribution
human_count = (daigt_df['label'] == 0).sum()
ai_count    = (daigt_df['label'] == 1).sum()
print(f"    Human (0) : {human_count:,}")
print(f"    AI    (1) : {ai_count:,}")
assert human_count > 0, "No human examples found"
assert ai_count    > 0, "No AI examples found"
 
# Check 5 — No empty strings
assert (daigt_df['text'].str.strip() != '').all(), \
    "Empty string found in text column after cleaning"
print("  ✓ No empty strings in text column")
 
# Check 6 — Minimum word count satisfied
assert (daigt_df['word_count'] >= MIN_WORD_COUNT).all(), \
    f"Text below minimum word count ({MIN_WORD_COUNT}) found"
print(f"  ✓ All texts have >= {MIN_WORD_COUNT} words")
 
# Check 7 — Dataset column correct
assert (daigt_df['dataset'] == DATASET_NAME).all(), \
    "Dataset column contains unexpected values"
print(f"  ✓ Dataset column correctly set to '{DATASET_NAME}'")
 
# Check 8 — No duplicates
assert daigt_df['text'].duplicated().sum() == 0, \
    "Duplicate texts remain after deduplication step"
print("  ✓ No duplicate texts")
 
print("\n  ALL DAIGT ASSERTIONS PASSED ✓")


STEP 10: Running data quality assertions (DAIGT)
  ✓ All required columns present
  ✓ No null values in text or label columns
  ✓ All labels are 0 or 1
  ✓ Both classes (0=Human, 1=AI) present
    Human (0) : 27,364
    AI    (1) : 46,196
  ✓ No empty strings in text column
  ✓ All texts have >= 10 words
  ✓ Dataset column correctly set to 'DAIGT'
  ✓ No duplicate texts

  ALL DAIGT ASSERTIONS PASSED ✓


In [102]:

# =============================================================================
# SECTION 11 — Save Cleaned DAIGT
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 11: Saving cleaned DAIGT")
print("=" * 60)
 
daigt_df.to_csv(DAIGT_OUT_FILE, index=False, encoding='utf-8-sig')
print(f"  Saved {len(daigt_df):,} rows to : {DAIGT_OUT_FILE}")
 
print(f"\n  DAIGT label distribution:")
daigt_counts = daigt_df['label'].value_counts().sort_index()
for label, count in daigt_counts.items():
    name = "Human" if label == 0 else "AI"
    print(f"    {label} ({name}) : {count:,}  ({count/len(daigt_df)*100:.1f}%)")


STEP 11: Saving cleaned DAIGT
  Saved 73,560 rows to : /Users/yashaswini11/Desktop/Team_project/project/daigt_clean.csv

  DAIGT label distribution:
    0 (Human) : 27,364  (37.2%)
    1 (AI) : 46,196  (62.8%)


In [103]:
 
# =============================================================================
# SECTION 12 — Merge HC3 + DAIGT
# =============================================================================
# We now stack the two cleaned datasets into one unified dataframe.
# The 'dataset' column (HC3 vs DAIGT) and 'source' column are preserved
# so we can always split them back apart for cross-dataset evaluation.
#
# Column alignment:
#   Both dataframes have: text | label | source | dataset | word_count
#   pd.concat with axis=0 stacks them vertically.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 12: Merging HC3 + DAIGT")
print("=" * 60)
 
# Ensure column order is identical before merging
col_order = ['text', 'label', 'source', 'dataset', 'word_count']
hc3_df    = hc3_df[col_order]
daigt_df  = daigt_df[col_order]
 
merged_df = pd.concat([hc3_df, daigt_df], axis=0, ignore_index=True)
 
print(f"  HC3 rows         : {len(hc3_df):,}")
print(f"  DAIGT rows       : {len(daigt_df):,}")
print(f"  Merged total     : {len(merged_df):,}")
print(f"\n  Merged label distribution:")
merged_counts = merged_df['label'].value_counts().sort_index()
for label, count in merged_counts.items():
    name = "Human" if label == 0 else "AI"
    print(f"    {label} ({name}) : {count:,}  ({count/len(merged_df)*100:.1f}%)")
 
print(f"\n  Rows by dataset:")
print(merged_df['dataset'].value_counts().to_string())
 
print(f"\n  Rows by source/domain:")
print(merged_df['source'].value_counts().to_string())


STEP 12: Merging HC3 + DAIGT
  HC3 rows         : 78,733
  DAIGT rows       : 73,560
  Merged total     : 152,293

  Merged label distribution:
    0 (Human) : 79,889  (52.5%)
    1 (AI) : 72,404  (47.5%)

  Rows by dataset:
dataset
HC3      78733
DAIGT    73560

  Rows by source/domain:
source
reddit_eli5                               61572
persuade_corpus                           25993
finance                                    8368
persuade_finetuned_llamas                  8306
Mistral7B_CME_v7                           4890
open_qa                                    4639
llama_falcon_v3_llama_70b                  3500
llama_falcon_v3_falcon_180b                3493
Intel-neural-chat-7b-v3-1_LLMEssays_v1     3309
medicine                                   2542
chat_gpt_moth                              2421
mistral7binstruct_v2                       2421
mistral7binstruct_v1                       2420
llama2_chat                                2417
wiki_csai                      

In [104]:
 
# =============================================================================
# SECTION 13 — Cross-Dataset Deduplication
# =============================================================================
# After merging, a text that appears in both HC3 AND DAIGT would cause
# data leakage — the model could see the same text in training and testing.
#
# We remove cross-dataset duplicates after merging so the deduplication
# is applied globally across both sources at once.
#
# keep='first' retains the HC3 version when a duplicate spans both datasets
# (since HC3 appears first in the merged frame).
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 13: Cross-dataset deduplication")
print("=" * 60)
 
before = len(merged_df)
merged_df = merged_df.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)
after = len(merged_df)
 
print(f"  Rows before cross-dataset dedup : {before:,}")
print(f"  Rows after                       : {after:,}")
print(f"  Cross-dataset duplicates removed : {before - after:,}")


STEP 13: Cross-dataset deduplication
  Rows before cross-dataset dedup : 152,293
  Rows after                       : 152,293
  Cross-dataset duplicates removed : 0


In [105]:
 
# =============================================================================
# SECTION 14 — Class Balance Check and Optional Undersampling
# =============================================================================
# After merging, we check whether the combined dataset is sufficiently balanced.
#
# Why balance matters:
#   An 80/20 split (80% AI, 20% human) means a model that always predicts "AI"
#   gets 80% accuracy while being completely useless. Macro F1 penalises this
#   but training on heavily imbalanced data still biases learned weights.
#
# Strategy:
#   If the minority class ratio drops below MIN_CLASS_RATIO (default 0.35),
#   we randomly undersample the majority class to restore balance.
#   We do NOT oversample (SMOTE etc.) because text oversampling with naive
#   methods produces low-quality synthetic samples.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 14: Class balance check and undersampling if needed")
print("=" * 60)
 
label_counts    = merged_df['label'].value_counts()
minority_count  = label_counts.min()
majority_count  = label_counts.max()
minority_ratio  = minority_count / len(merged_df)
majority_label  = label_counts.idxmax()
minority_label  = label_counts.idxmin()
 
print(f"  Minority class  : {minority_label}  ({minority_count:,} rows)")
print(f"  Majority class  : {majority_label}  ({majority_count:,} rows)")
print(f"  Minority ratio  : {minority_ratio:.3f}  (threshold: {MIN_CLASS_RATIO})")
 
if minority_ratio < MIN_CLASS_RATIO:
    print(f"\n  Minority ratio {minority_ratio:.3f} < threshold {MIN_CLASS_RATIO}.")
    print(f"  Applying random undersampling of majority class...")
 
    minority_df = merged_df[merged_df['label'] == minority_label]
    majority_df = merged_df[merged_df['label'] == majority_label]
 
    # Undersample majority to match minority count
    majority_sampled = majority_df.sample(n=len(minority_df), random_state=RANDOM_SEED)
 
    merged_df = pd.concat([minority_df, majority_sampled], axis=0, ignore_index=True)
    merged_df = merged_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
 
    print(f"  Rows after undersampling : {len(merged_df):,}")
    print(f"  New label distribution:")
    for label, count in merged_df['label'].value_counts().sort_index().items():
        name = "Human" if label == 0 else "AI"
        print(f"    {label} ({name}) : {count:,}  ({count/len(merged_df)*100:.1f}%)")
else:
    print(f"\n  Minority ratio {minority_ratio:.3f} >= threshold {MIN_CLASS_RATIO}.")
    print("  Class balance is acceptable — no undersampling needed.")
 


STEP 14: Class balance check and undersampling if needed
  Minority class  : 1  (72,404 rows)
  Majority class  : 0  (79,889 rows)
  Minority ratio  : 0.475  (threshold: 0.35)

  Minority ratio 0.475 >= threshold 0.35.
  Class balance is acceptable — no undersampling needed.


In [106]:
# =============================================================================
# SECTION 15 — Save Merged Dataset
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 15: Saving merged dataset")
print("=" * 60)
 
merged_df.to_csv(MERGED_FILE, index=False, encoding='utf-8-sig')
print(f"  Saved {len(merged_df):,} rows to : {MERGED_FILE}")


STEP 15: Saving merged dataset
  Saved 152,293 rows to : /Users/yashaswini11/Desktop/Team_project/project/merged.csv


In [107]:

# =============================================================================
# SECTION 16 — Stratified Train / Val / Test Split
# =============================================================================
# We split AFTER merging and AFTER balance checks — never before.
# Splitting before would risk:
#   a) different class ratios across splits
#   b) texts from the same source concentrating in one split
#
# Stratified splitting (stratify=label) guarantees that each split contains
# the same proportion of Human and AI labels as the full merged dataset.
#
# Split ratios: 70% train | 15% val | 15% test
# The test set goes in a drawer after this step — it is NOT used for any
# model development, tuning, or decision-making. Only the final evaluation
# at the very end of the project opens it.
#
# We perform the split in two steps:
#   Step 1: Separate test set   → 85% temp  |  15% test
#   Step 2: Separate val set    → 70% train |  15% val  (from the 85% temp)
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 16: Stratified train / val / test split")
print("=" * 60)
 
# Carve out the test set first
# test_size = TEST_RATIO (0.15) of the full merged dataset
temp_df, test_df = train_test_split(
    merged_df,
    test_size=TEST_RATIO,
    stratify=merged_df['label'],
    random_state=RANDOM_SEED
)
 
# Split the remaining 85% into train (70%) and val (15%)
# We need val as a fraction of the temp (85%) set, not the full dataset.
# val_fraction_of_temp = VAL_RATIO / (1 - TEST_RATIO) = 0.15 / 0.85 ≈ 0.176
val_fraction_of_temp = VAL_RATIO / (1 - TEST_RATIO)
 
train_df, val_df = train_test_split(
    temp_df,
    test_size=val_fraction_of_temp,
    stratify=temp_df['label'],
    random_state=RANDOM_SEED
)
 
# Reset indices on all three splits
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
 
print(f"  Total merged rows : {len(merged_df):,}")
print(f"  Train rows        : {len(train_df):,}  ({len(train_df)/len(merged_df)*100:.1f}%)")
print(f"  Val rows          : {len(val_df):,}   ({len(val_df)/len(merged_df)*100:.1f}%)")
print(f"  Test rows         : {len(test_df):,}   ({len(test_df)/len(merged_df)*100:.1f}%)")

# Verify each split has both classes and correct proportions
print(f"\n  Train label distribution:")
for label, count in train_df['label'].value_counts().sort_index().items():
    name = "Human" if label == 0 else "AI"
    print(f"    {label} ({name}) : {count:,}  ({count/len(train_df)*100:.1f}%)")
 
print(f"\n  Val label distribution:")
for label, count in val_df['label'].value_counts().sort_index().items():
    name = "Human" if label == 0 else "AI"
    print(f"    {label} ({name}) : {count:,}  ({count/len(val_df)*100:.1f}%)")
 
print(f"\n  Test label distribution:")
for label, count in test_df['label'].value_counts().sort_index().items():
    name = "Human" if label == 0 else "AI"
    print(f"    {label} ({name}) : {count:,}  ({count/len(test_df)*100:.1f}%)")


STEP 16: Stratified train / val / test split
  Total merged rows : 152,293
  Train rows        : 106,605  (70.0%)
  Val rows          : 22,844   (15.0%)
  Test rows         : 22,844   (15.0%)

  Train label distribution:
    0 (Human) : 55,923  (52.5%)
    1 (AI) : 50,682  (47.5%)

  Val label distribution:
    0 (Human) : 11,983  (52.5%)
    1 (AI) : 10,861  (47.5%)

  Test label distribution:
    0 (Human) : 11,983  (52.5%)
    1 (AI) : 10,861  (47.5%)


In [108]:
# =============================================================================
# SECTION 17 — Split Integrity Assertions
# =============================================================================
# These assertions catch data leakage between splits — the most dangerous
# bug in a machine learning pipeline because it silently inflates all metrics.
#
# The most critical check: no text appears in more than one split.
# If the same text is in both train and test, the model has already "seen"
# the answer during training.
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 17: Split integrity assertions")
print("=" * 60)
 
# Check 1 — Row counts add up
total_split_rows = len(train_df) + len(val_df) + len(test_df)
assert total_split_rows == len(merged_df), \
    f"Row count mismatch: splits sum to {total_split_rows}, merged has {len(merged_df)}"
print("  ✓ Row counts sum correctly to merged total")
 
# Check 2 — No text overlap between train and test (most critical)
train_texts = set(train_df['text'])
val_texts   = set(val_df['text'])
test_texts  = set(test_df['text'])
 
train_test_overlap = train_texts & test_texts
assert len(train_test_overlap) == 0, \
    f"DATA LEAKAGE: {len(train_test_overlap)} texts appear in both train and test"
print("  ✓ Zero text overlap between train and test")
 
# Check 3 — No text overlap between train and val
train_val_overlap = train_texts & val_texts
assert len(train_val_overlap) == 0, \
    f"DATA LEAKAGE: {len(train_val_overlap)} texts appear in both train and val"
print("  ✓ Zero text overlap between train and val")
 
# Check 4 — No text overlap between val and test
val_test_overlap = val_texts & test_texts
assert len(val_test_overlap) == 0, \
    f"DATA LEAKAGE: {len(val_test_overlap)} texts appear in both val and test"
print("  ✓ Zero text overlap between val and test")
 
# Check 5 — Both classes in every split
for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert split_df['label'].nunique() == 2, \
        f"Split '{split_name}' is missing one of the label classes"
print("  ✓ Both classes present in all three splits")
 
# Check 6 — Stratification: class ratio within 2 percentage points across splits
target_ratio = merged_df['label'].mean()   # expected proportion of AI labels
for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split_ratio = split_df['label'].mean()
    assert abs(split_ratio - target_ratio) < 0.02, \
        f"Stratification off in {split_name}: " \
        f"expected AI ratio ~{target_ratio:.3f}, got {split_ratio:.3f}"
print(f"  ✓ AI label ratio consistent across splits (target: {target_ratio:.3f})")
 
print("\n  ALL SPLIT INTEGRITY ASSERTIONS PASSED ✓")


STEP 17: Split integrity assertions
  ✓ Row counts sum correctly to merged total
  ✓ Zero text overlap between train and test
  ✓ Zero text overlap between train and val
  ✓ Zero text overlap between val and test
  ✓ Both classes present in all three splits
  ✓ AI label ratio consistent across splits (target: 0.475)

  ALL SPLIT INTEGRITY ASSERTIONS PASSED ✓


In [109]:

# =============================================================================
# SECTION 18 — Save Splits
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 18: Saving splits")
print("=" * 60)
 
train_df.to_csv(TRAIN_FILE, index=False, encoding='utf-8-sig')
val_df.to_csv(VAL_FILE,   index=False, encoding='utf-8-sig')
test_df.to_csv(TEST_FILE,  index=False, encoding='utf-8-sig')
 
print(f"  Saved train : {TRAIN_FILE}  ({len(train_df):,} rows)")
print(f"  Saved val   : {VAL_FILE}    ({len(val_df):,} rows)")
print(f"  Saved test  : {TEST_FILE}   ({len(test_df):,} rows)")
 
print("\n  !! IMPORTANT: test.csv is now locked.")
print("  Do NOT use test.csv during model development, tuning, or debugging.")
print("  It is only opened once — at the very end for final KPI evaluation.")
 
 



STEP 18: Saving splits
  Saved train : /Users/yashaswini11/Desktop/Team_project/project/train.csv  (106,605 rows)
  Saved val   : /Users/yashaswini11/Desktop/Team_project/project/val.csv    (22,844 rows)
  Saved test  : /Users/yashaswini11/Desktop/Team_project/project/test.csv   (22,844 rows)

  !! IMPORTANT: test.csv is now locked.
  Do NOT use test.csv during model development, tuning, or debugging.
  It is only opened once — at the very end for final KPI evaluation.


In [110]:

# =============================================================================
# SECTION 19 — Final Summary
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 19: Final pipeline summary")
print("=" * 60)
 
print(f"""
  Files produced:
    {DAIGT_OUT_FILE:<40} {len(daigt_df):>8,} rows  (cleaned DAIGT only)
    {MERGED_FILE:<40} {len(merged_df):>8,} rows  (HC3 + DAIGT combined)
    {TRAIN_FILE:<40} {len(train_df):>8,} rows  (70% — for model training)
    {VAL_FILE:<40} {len(val_df):>8,} rows  (15% — for tuning and early stopping)
    {TEST_FILE:<40} {len(test_df):>8,} rows  (15% — locked until final evaluation)
 
  Next steps:
    Run feature_engineering.py to build:
      - TF-IDF sparse matrices from train/val/test
      - Stylometric feature vectors
      - GloVe tokenised sequences for the Hybrid CNN
      - RoBERTa tokenised inputs for the transformer model
""")
 
print("=" * 60)
print("DAIGT data engineering complete.")
print("Next step: run feature_engineering.py")
print("=" * 60)
 


STEP 19: Final pipeline summary

  Files produced:
    /Users/yashaswini11/Desktop/Team_project/project/daigt_clean.csv   73,560 rows  (cleaned DAIGT only)
    /Users/yashaswini11/Desktop/Team_project/project/merged.csv  152,293 rows  (HC3 + DAIGT combined)
    /Users/yashaswini11/Desktop/Team_project/project/train.csv  106,605 rows  (70% — for model training)
    /Users/yashaswini11/Desktop/Team_project/project/val.csv   22,844 rows  (15% — for tuning and early stopping)
    /Users/yashaswini11/Desktop/Team_project/project/test.csv   22,844 rows  (15% — locked until final evaluation)

  Next steps:
    Run feature_engineering.py to build:
      - TF-IDF sparse matrices from train/val/test
      - Stylometric feature vectors
      - GloVe tokenised sequences for the Hybrid CNN
      - RoBERTa tokenised inputs for the transformer model

DAIGT data engineering complete.
Next step: run feature_engineering.py
